# 3장. 데이터 연습 (pandas · 통계 · 크롤링)

고객지원 문의 1,015건(가짜 데이터)으로 **정제 → 집계 → 해석** 흐름을 그대로 따라가요.
크롤링은 인터넷 없이 되도록 HTML·JSON을 문자열로 넣어 두었어요. 실제 사이트에서는 `requests.get(url, timeout=10).text`만 바꾸면 돼요.

> 분석 흐름: ① 질문 정하기 → ② 수집 → ③ 정제 → ④ 탐색(EDA) → ⑤ 해석·전달

In [ ]:
# ▶ 먼저 이 셀을 실행하세요 (채점 도우미)
import os
STRICT = os.environ.get("PRACTICE_STRICT") == "1"
___ = None  # 빈칸 표시. 이 자리를 채우면 돼요

def check(name, cond, hint=""):
    """cond가 참이면 ✅, 아니면 ❌ (cond는 값이나 lambda)"""
    try:
        ok = bool(cond() if callable(cond) else cond)
    except Exception as e:
        ok, hint = False, f"{type(e).__name__}: {e}"
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"  → {hint}"))
    if STRICT and not ok:
        raise AssertionError(name)

## 0. 데이터 만들기 (그냥 실행)
**질문**: "어떤 문의가 상담사로 넘어가고(`handoff`), 응답은 얼마나 걸리나(`sec`)?"

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 1000
types = ["배송", "환불", "교환", "상품문의", "결제"]
handoff_p = {"배송": 0.05, "환불": 0.6, "교환": 0.5, "상품문의": 0.15, "결제": 0.3}
base = pd.DataFrame({
    "ticket_id": np.arange(1, n + 1),
    "mall": rng.choice(list("ABCDE"), n, p=[.3, .25, .2, .15, .1]),
    "type": rng.choice(types, n, p=[.4, .2, .1, .2, .1]),
    "created_at": pd.Timestamp("2026-10-01") + pd.to_timedelta(rng.integers(0, 30 * 24 * 60, n), unit="min"),
})
base["handoff"] = (rng.random(n) < base["type"].map(handoff_p)).astype(int)
base["sec"] = np.round(rng.lognormal(0.3, 0.6, n) + base["handoff"] * 1.5, 2)
base.loc[rng.choice(n, 40, replace=False), "sec"] = np.nan              # 결측치 40개

raw = (pd.concat([base, base.sample(15, random_state=1)])               # 중복 15개
         .sample(frac=1, random_state=2).reset_index(drop=True))
raw["created_at"] = raw["created_at"].dt.strftime("%Y-%m-%d %H:%M")      # 일부러 문자열로

malls = pd.DataFrame({"mall": list("ABCD"), "plan": ["프로", "베이직", "프로", "베이직"]})  # E몰 정보는 없음
raw.head()

## 3-1. 정제: 결측치와 중복
- `n_missing`: `sec`의 결측치 개수
- `n_dup`: 완전히 똑같은 행(중복) 개수
- `df`: 중복을 지운 뒤, `sec` 결측치를 **중앙값**으로 채운 표 (원본 `raw`는 그대로 두기)

> 응답 시간처럼 꼬리가 긴 값은 평균보다 중앙값으로 채우는 게 덜 왜곡돼요.

In [ ]:
raw.info()
n_missing = raw["sec"].isna().sum()
n_dup = raw.duplicated().sum()
df = raw.drop_duplicates().copy()
df["sec"] = df["sec"].fillna(df["sec"].median())

In [ ]:
check("결측치 개수", lambda: n_missing == sum(v != v for v in raw["sec"]))
check("중복 15개", lambda: n_dup == 15)
check("중복 제거 후 1000행", lambda: len(df) == 1000)
check("결측치 0", lambda: df["sec"].isna().sum() == 0)
check("중앙값으로 채움", lambda: np.isclose(df.loc[raw.drop_duplicates()["sec"].isna().values, "sec"],
                                         raw.drop_duplicates()["sec"].median()).all())

## 3-2. 그룹 집계 (SQL의 GROUP BY)
문의 유형(`type`)별로 `cnt`(건수), `handoff_rate`(넘어간 비율), `avg_sec`(평균 응답초)를 구해 **건수 내림차순**으로 정렬해요.

In [ ]:
summary = (df.groupby("type")
             .agg(cnt=("ticket_id", "size"), handoff_rate=("handoff", "mean"), avg_sec=("sec", "mean"))
             .sort_values("cnt", ascending=False))
summary.round(2)

In [ ]:
check("열 이름", lambda: list(summary.columns) == ["cnt", "handoff_rate", "avg_sec"])
check("건수 내림차순", lambda: summary["cnt"].is_monotonic_decreasing)
check("건수 합 1000", lambda: summary["cnt"].sum() == 1000)
check("환불 넘어감 비율", lambda: np.isclose(summary.loc["환불", "handoff_rate"], df[df.type == "환불"].handoff.mean()))
check("배송 평균 응답", lambda: np.isclose(summary.loc["배송", "avg_sec"], df[df.type == "배송"].sec.mean()))

In [ ]:
top2_handoff = summary["handoff_rate"].nlargest(2).index.tolist()

In [ ]:
check("환불·교환이 가장 많이 넘어감", lambda: list(top2_handoff) == ["환불", "교환"])

## 3-3. 교차표와 JOIN
- `ct`: 몰 × 문의유형 표를 **행 기준 비율**(각 몰 안에서 유형 비율, 행 합 = 1)로
- `merged`: `df`에 `malls`의 `plan`을 붙이되 **행이 사라지면 안 돼요** (E몰은 `"미가입"`으로 채우기)
- `by_plan`: 요금제별 `handoff` 비율 (Series)

In [ ]:
ct = pd.crosstab(df["mall"], df["type"], normalize="index")
merged = df.merge(malls, on="mall", how="left")
merged["plan"] = merged["plan"].fillna("미가입")
by_plan = merged.groupby("plan")["handoff"].mean()
ct.round(2), by_plan.round(3)

In [ ]:
check("교차표 모양 5몰×5유형", lambda: ct.shape == (5, 5))
check("행 합 = 1", lambda: np.allclose(ct.sum(axis=1), 1))
check("LEFT JOIN 후에도 1000행", lambda: len(merged) == 1000)
check("E몰은 미가입", lambda: set(merged.loc[merged.mall == "E", "plan"]) == {"미가입"})
check("요금제 3종", lambda: set(by_plan.index) == {"프로", "베이직", "미가입"})

## 3-4. 날짜 다루기
`created_at`이 문자열이에요. 날짜로 바꾸고 **일별 건수**(`daily`)와 **가장 바쁜 날**(`busiest`), **요일별 평균 건수**를 구해요.

In [ ]:
df["created_at"] = pd.to_datetime(df["created_at"])
daily = df.set_index("created_at").resample("D").size()
busiest = daily.idxmax()
by_weekday = daily.groupby(daily.index.day_name()).mean().round(1)
daily.tail(3), busiest, by_weekday

In [ ]:
check("날짜 자료형", lambda: pd.api.types.is_datetime64_any_dtype(df["created_at"]))
check("30일", lambda: len(daily) == 30 and daily.sum() == 1000)
_vc = raw.drop_duplicates()["created_at"].str[:10].value_counts()
check("가장 바쁜 날", lambda: _vc[str(busiest.date())] == _vc.max())

## 3-5. 평균 vs 중앙값 vs p95
응답 시간처럼 **꼬리가 긴** 데이터는 평균이 느린 값 몇 개에 끌려가요.
`stats(s)`가 `{"mean", "median", "p95"}`를 돌려주게 만들고, 300초짜리 이상값 하나를 넣었을 때 무엇이 크게 변하는지 봐요.

In [ ]:
def stats(s):
    return {"mean": s.mean(), "median": s.median(), "p95": s.quantile(0.95)}

before = stats(df["sec"])
after = stats(pd.concat([df["sec"], pd.Series([300.0])]))
pd.DataFrame({"before": before, "after": after}).round(3)

In [ ]:
check("p95", lambda: np.isclose(before["p95"], np.percentile(df["sec"], 95)))
check("오른쪽 꼬리: 평균 > 중앙값", lambda: before["mean"] > before["median"])
check("이상값 1개에 평균은 크게 흔들림", lambda: after["mean"] - before["mean"] > 0.25)
check("중앙값은 거의 그대로", lambda: abs(after["median"] - before["median"]) < 0.01)

## 3-6. A/B 테스트 (순열 검정)
상담 화면에 "FAQ 추천" 버튼을 넣은 B안이 **자체 해결률**을 올렸을까요?
p-value를 **순열 검정**으로 직접 구해요. 아이디어: "A/B 구분이 의미 없다면, 라벨을 무작위로 섞어도 지금 정도 차이는 자주 나와야 한다."

1. 두 그룹을 합친다
2. 무작위로 섞어 앞 nA개를 A, 나머지를 B라고 치고 차이를 잰다 (5,000번)
3. 섞은 차이의 절댓값이 **실제 차이 이상**인 비율 = p-value

In [ ]:
def perm_test(a, b, n_iter=5000, seed=0):
    rng = np.random.default_rng(seed)
    observed = b.mean() - a.mean()
    pooled = np.concatenate([a, b])
    count = 0
    for _ in range(n_iter):
        shuffled = rng.permutation(pooled)
        diff = shuffled[len(a):].mean() - shuffled[:len(a)].mean()
        if abs(diff) >= abs(observed):
            count += 1
    return observed, count / n_iter

In [ ]:
big_A = np.array([1] * 600 + [0] * 1400)     # 2000명 중 600명 해결 (30%)
big_B = np.array([1] * 720 + [0] * 1280)     # 2000명 중 720명 해결 (36%)
small_A = np.array([1] * 12 + [0] * 28)      # 40명 중 12명 (30%)
small_B = np.array([1] * 14 + [0] * 26)      # 40명 중 14명 (35%)

d1, p1 = perm_test(big_A, big_B)
d2, p2 = perm_test(small_A, small_B)
print(f"표본 큼: 차이 {d1:+.3f}, p = {p1:.4f}")
print(f"표본 작음: 차이 {d2:+.3f}, p = {p2:.4f}")
check("큰 표본: 0.05 미만 (우연이라 보기 어려움)", lambda: p1 < 0.05 < p2)
check("작은 표본: 비슷한 차이여도 0.05 이상", lambda: p2 > 0.05)

In [ ]:
stat_quiz = {
    "아이스크림 판매량과 익사 사고가 함께 늘었다 → 아이스크림이 원인이다 (True/False)": False,  # 상관 ≠ 인과 (더위가 공통 원인)
    "원인을 확인하려면 무작위로 나눠 한쪽만 바꾸는 ___ 를 한다":                      "A/B 테스트",
    "상담사로 넘어온 로그만 보고 '전체 고객 만족도'를 말하면 생기는 문제":              "표본 편향",
    "날짜별 추이":        "선",
    "유형별 건수 비교":    "막대",
    "응답 시간 분포":      "히스토그램",
    "응답 시간과 대화 턴 수의 관계": "산점도",
}

In [ ]:
sq = list(stat_quiz.values())
check("상관 ≠ 인과", lambda: sq[0] is False)
check("A/B 테스트", lambda: str(sq[1]).replace(" ", "").upper() in ("A/B테스트", "AB테스트", "A/BTEST"))
check("표본 편향", lambda: sq[2] == "표본 편향")
check("그래프 고르기", lambda: sq[3:] == ["선", "막대", "히스토그램", "산점도"])

## 3-7. 크롤링 ① HTML에 내용이 바로 있을 때 (BeautifulSoup)
FAQ 영역(`div.faq`)의 질문/답만 뽑아요. **광고 영역은 빼야** 해요. 카테고리는 `section`의 `data-cat` 속성에 있어요.
결과: `[{"cat": ..., "q": ..., "a": ...}, ...]`

In [ ]:
from bs4 import BeautifulSoup
html = """
<html><body>
<div class="faq">
  <section data-cat="배송">
    <h2>배송</h2>
    <ul>
      <li><h3> 배송은 얼마나 걸리나요? </h3><p>결제 후 2~3일</p></li>
      <li><h3>배송비는 얼마인가요?</h3><p>3만원 이상 무료</p></li>
    </ul>
  </section>
  <section data-cat="반품">
    <h2>반품</h2>
    <ul><li><h3>반품 기한은?</h3><p>수령 후 7일 이내</p></li></ul>
  </section>
</div>
<div class="ad"><ul><li><h3>오늘만 특가!</h3><p>지금 구매하세요</p></li></ul></div>
</body></html>
"""
soup = BeautifulSoup(html, "html.parser")

In [ ]:
faq = []
for sec in soup.select("div.faq section"):
    for li in sec.select("li"):
        faq.append({"cat": sec["data-cat"], "q": li.h3.get_text(strip=True), "a": li.p.get_text(strip=True)})
faq

In [ ]:
check("FAQ 3개 (광고 제외, 공백 제거)", lambda: faq == [
    {"cat": "배송", "q": "배송은 얼마나 걸리나요?", "a": "결제 후 2~3일"},
    {"cat": "배송", "q": "배송비는 얼마인가요?", "a": "3만원 이상 무료"},
    {"cat": "반품", "q": "반품 기한은?", "a": "수령 후 7일 이내"}])

### 표를 DataFrame으로
상품 표를 읽어 `name`, `price`(**정수**), `soldout`(품절이면 True) 열의 DataFrame을 만들어요.
가격 `"12,900원"` → `12900`, 품절은 `<tr class="soldout">`이에요.

In [ ]:
table_html = """
<table id="products">
  <tr><th>상품</th><th>가격</th></tr>
  <tr><td>반팔 티셔츠</td><td>12,900원</td></tr>
  <tr class="soldout"><td>청바지</td><td>39,000원</td></tr>
  <tr><td>양말 3켤레</td><td>5,500원</td></tr>
</table>
"""
tsoup = BeautifulSoup(table_html, "html.parser")

In [ ]:
rows = []
for tr in tsoup.select("#products tr")[1:]:
    name, price = [td.get_text(strip=True) for td in tr.select("td")]
    rows.append({"name": name,
                 "price": int(price.replace(",", "").replace("원", "")),
                 "soldout": "soldout" in (tr.get("class") or [])})
products = pd.DataFrame(rows)
products

In [ ]:
check("가격 정수", lambda: products["price"].tolist() == [12900, 39000, 5500])
check("품절 표시", lambda: products["soldout"].tolist() == [False, True, False])

## 3-8. 크롤링 ② JavaScript가 나중에 그리는 페이지
HTML을 받아 보면 내용이 **비어 있어요**. 브라우저 개발자도구 **Network** 탭에서 데이터를 주는 API(JSON)를 찾아 그걸 직접 부르는 게 가장 빠르고 안정적이에요.
아래 `fake_api(page)`가 그 API라고 치고, **`next`가 없을 때까지** 모든 페이지를 모아요.

In [ ]:
import json
spa_html = '<html><body><div id="app"></div><script src="/static/bundle.js"></script></body></html>'
print("HTML에서 찾은 li 개수:", len(BeautifulSoup(spa_html, "html.parser").select("li")))   # 0개!

_PAGES = {
    1: {"items": [{"q": "배송 조회", "a": "마이페이지"}, {"q": "배송지 변경", "a": "출고 전 가능"}, {"q": "해외 배송", "a": "불가"}], "next": 2},
    2: {"items": [{"q": "환불 기간", "a": "3~5일"}, {"q": "부분 환불", "a": "가능"}, {"q": "환불 계좌", "a": "결제 수단으로"}], "next": 3},
    3: {"items": [{"q": "회원 탈퇴", "a": "설정 메뉴"}], "next": None},
}
calls = []
def fake_api(page):
    """GET /api/faq?page=N 응답(문자열)이라고 생각하세요"""
    calls.append(page)
    return json.dumps(_PAGES[page], ensure_ascii=False)

In [ ]:
all_items, page = [], 1
while page is not None:
    data = json.loads(fake_api(page))
    all_items += data["items"]
    page = data["next"]
len(all_items)

In [ ]:
check("7개 모두 수집", lambda: len(all_items) == 7)
check("페이지 1→2→3 한 번씩만 호출", lambda: calls == [1, 2, 3])

## 3-9. 크롤링 예의: robots.txt · 재시도
### (1) robots.txt 읽기

In [ ]:
from urllib.robotparser import RobotFileParser
robots_txt = """
User-agent: *
Disallow: /admin
Disallow: /cart
Crawl-delay: 2
"""
rp = RobotFileParser()
rp.parse(robots_txt.splitlines())

In [ ]:
url = lambda p: "https://shop.example.com" + p
robots = {
    "/products/123 수집 가능?": rp.can_fetch("*", url("/products/123")),
    "/admin/users 수집 가능?":  rp.can_fetch("*", url("/admin/users")),
    "/cart 수집 가능?":         rp.can_fetch("*", url("/cart")),
    "요청 사이 쉬어야 할 초":    rp.crawl_delay("*"),
}

In [ ]:
check("robots.txt", lambda: list(robots.values()) == [True, False, False, 2])

### (2) 지수 백오프 재시도
네트워크는 가끔 실패해요. `fetch_with_retry`를 완성해요.
- 최대 `tries`번 시도, 성공하면 바로 결과 반환
- 실패하면 `base`, `base*2`, `base*4`, ... 초 쉬고 다시 (마지막 실패 뒤에는 쉬지 않음)
- 다 실패하면 마지막 예외를 그대로 `raise`
- 테스트를 위해 `sleep` 함수를 인자로 받아요 (실제로는 `time.sleep`)

In [ ]:
def fetch_with_retry(fetch, tries=3, base=0.5, sleep=None):
    for attempt in range(tries):
        try:
            return fetch()
        except Exception:
            if attempt == tries - 1:
                raise
            sleep(base * 2 ** attempt)

In [ ]:
def make_flaky(fail_times):
    state = {"n": 0}
    def fetch():
        state["n"] += 1
        if state["n"] <= fail_times:
            raise ConnectionError("timeout")
        return "<html>OK</html>"
    return fetch

slept = []
check("2번 실패 후 성공", lambda: fetch_with_retry(make_flaky(2), sleep=slept.append) == "<html>OK</html>")
check("0.5초, 1.0초 쉬었다", lambda: slept == [0.5, 1.0])

slept2 = []
def always_fail():
    try:
        fetch_with_retry(make_flaky(99), tries=3, sleep=slept2.append)
    except ConnectionError:
        return True
    return False
check("다 실패하면 예외", always_fail)
check("마지막 실패 뒤엔 안 쉼", lambda: slept2 == [0.5, 1.0])

## ✍️ 말로 설명해 보기 (30초)
1. 응답 시간을 평균 대신 p95로 보는 이유는?
2. JS로 그려지는 페이지는 어떻게 크롤링하나?
3. 크롤링할 때 지켜야 할 것은?

<details><summary>모범 답안</summary>

1. 응답 시간은 꼬리가 길어서 평균이 느린 몇 건에 끌려가요. 중앙값은 "보통 사용자", p95는 "느린 쪽 5% 경계"를 보여줘서 실제 체감과 최악에 가까운 경험을 같이 볼 수 있어요.
2. 먼저 개발자도구 Network 탭에서 데이터를 주는 JSON API를 찾아 직접 호출해요(빠르고 안정적). 그게 안 되면 Playwright·Selenium으로 실제 브라우저를 띄워 렌더링된 뒤 읽어요.
3. robots.txt·이용약관 확인, 요청 사이 쉬기, 타임아웃·재시도, 로그인 뒤 개인정보는 수집하지 않기, 공식 API가 있으면 API 먼저. 구조가 바뀌면 깨지므로 수집 개수를 로그로 남겨요.
</details>